# Job Market Intelligence – Data Cleaning

This notebook cleans and prepares the No Fluff Jobs dataset for exploratory data analysis and machine learning.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

RAW_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "nofluff_it_jobs.csv"
)

df = pd.read_csv(RAW_PATH)

raw_row_count = len(df)

expired_offer_mask = (
    df["company"]
    .fillna("")
    .str.strip()
    .eq("Zobacz profil firmy")
)

df = (
    df.loc[~expired_offer_mask]
    .copy()
    .reset_index(drop=True)
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Raw rows:", raw_row_count)
print("Expired offers removed:", expired_offer_mask.sum())
print("Dataset shape after removal:", df.shape)

PROJECT_ROOT: G:\pandas\job_market_intelligence
Raw rows: 3360
Expired offers removed: 0
Dataset shape after removal: (3360, 25)


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3360 entries, 0 to 3359
Data columns (total 25 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   job_id                3360 non-null   object 
 1   url                   3360 non-null   object 
 2   title                 3360 non-null   object 
 3   category              3360 non-null   object 
 4   experience            3332 non-null   object 
 5   experience_years_min  2031 non-null   float64
 6   workplace             3070 non-null   object 
 7   job_locations         2094 non-null   object 
 8   company               3360 non-null   object 
 9   company_size          2474 non-null   object 
 10  company_founded       2474 non-null   float64
 11  company_locations     2474 non-null   object 
 12  salary_min            2353 non-null   float64
 13  salary_max            2353 non-null   float64
 14  salary_currency       2353 non-null   object 
 15  salary_period        

## 1. Missing values

Check the number and percentage of missing values in each column.

In [3]:
missing_values = (
    df.isna()
    .sum()
    .to_frame("missing_count")
)

missing_values["missing_percent"] = (
    missing_values["missing_count"]
    / len(df)
    * 100
).round(2)

missing_values = missing_values.sort_values(
    "missing_percent",
    ascending=False
)

missing_values

,missing_count,missing_percent
experience_years_min,1329,39.55
nice_to_have,1318,39.23
job_locations,1266,37.68
salary_period,1008,30.00
salary_max,1007,29.97
salary_min,1007,29.97
salary_currency,1007,29.97
company_locations,886,26.37
company_size,886,26.37
company_founded,886,26.37


## 2. Duplicates

Check for duplicated rows, job IDs, and job URLs.

In [4]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate job IDs:", df["job_id"].duplicated().sum())
print("Duplicate URLs:", df["url"].duplicated().sum())

Duplicate rows: 0
Duplicate job IDs: 0
Duplicate URLs: 0


## 3. Categorical values

Inspect the most important categorical columns and identify inconsistent or unexpected values.

In [5]:
columns_to_check = [
    "experience",
    "workplace",
    "category",
    "contract_type",
    "salary_currency",
]

for col in columns_to_check:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).head(30))


--- experience ---
experience
Senior    1750
Mid       1296
Junior     185
Expert     101
NaN         28
Name: count, dtype: int64

--- workplace ---
workplace
Hybrid    1880
Remote    1180
NaN        290
Onsite      10
Name: count, dtype: int64

--- category ---
category
Backend                  625
Data                     430
Testing                  268
DevOps                   264
Fullstack                253
AI                       187
ERP                      175
Project Manager          168
Security                 155
Architecture             153
Support                   91
Product Management        81
Frontend                  74
Business Analysis         73
Inne IT                   57
Sys. Administrator        55
Embedded                  47
Mobile                    46
Design                    33
Business Intelligence     24
Finanse                   21
Sprzedaż                  20
Agile                     13
HR                        13
Automatyka                 8
T

## 4. Numerical values

Inspect numerical columns and check for invalid or unexpected values.

In [6]:
numerical_columns = [
    "experience_years_min",
    "company_founded",
    "salary_min",
    "salary_max",
]

df[numerical_columns].describe()

,experience_years_min,company_founded,salary_min,salary_max
count,2031.000000,2474.000000,2353.000000,2353.000000
mean,4.759232,2000.374697,11040.754781,14888.764981
std,2.030951,30.765603,32924.854321,45881.579082
min,1.000000,1845.000000,28.000000,32.000000
25%,3.000000,1996.000000,135.000000,160.000000
50%,5.000000,2007.000000,500.000000,650.000000
75%,5.000000,2014.000000,15000.000000,21000.000000
max,20.000000,2026.000000,350700.000000,474400.000000


In [7]:
current_year = pd.Timestamp.today().year

print(
    "Salary min > salary max:",
    (df["salary_min"] > df["salary_max"]).sum()
)

print(
    "Salary <= 0:",
    (
        (df["salary_min"] <= 0)
        | (df["salary_max"] <= 0)
    ).sum()
)

print(
    f"Company founded > {current_year}:",
    (df["company_founded"] > current_year).sum()
)

print(
    "Experience years < 0:",
    (df["experience_years_min"] < 0).sum()
)


Salary min > salary max: 0
Salary <= 0: 0
Company founded > 2026: 0
Experience years < 0: 0


In [8]:
df[["valid_until", "start_date", "scraped_at"]].head(20)

,valid_until,start_date,scraped_at
0,20.09.2026,2026-09-07,2026-08-24 01:15:58
1,11.09.2026,ASAP,2026-08-24 01:16:02
2,06.09.2026,ASAP,2026-08-24 01:16:05
3,03.09.2026,ASAP,2026-08-24 01:16:09
4,16.09.2026,ASAP,2026-08-24 01:16:12
5,17.09.2026,ASAP,2026-08-24 01:16:16
6,31.08.2026,ASAP,2026-08-24 01:16:19
7,28.08.2026,ASAP,2026-08-24 01:16:23
8,15.09.2026,NaN,2026-08-24 01:16:26
9,29.08.2026,ASAP,2026-08-24 01:16:30


In [9]:
start_date_check = (
    df["start_date"]
    .dropna()
    .astype(str)
    .str.len()
)

start_date_check.describe()

count    3192.000000
mean        4.530075
std         1.703050
min         4.000000
25%         4.000000
50%         4.000000
75%         4.000000
max        10.000000
Name: start_date, dtype: float64

In [10]:
df.loc[
    df["start_date"]
    .fillna("")
    .astype(str)
    .str.len()
    .sort_values(ascending=False)
    .head(20)
    .index,
    ["title", "start_date"]
]

,title,start_date
664,DevSecOps Engineer,2026-10-01
3152,System Analyst,2026-08-10
1197,"JavaScript (React, Typescript)/ Mashup Developer",2026-06-17
546,Data Scientist,2026-06-23
176,"Automation, Condition Monitoring & Digital Sol...",2026-08-03
1089,IT Engineer to perform Single Sign-On Onboarding,2026-08-01
0,1st Line Analyst with English,2026-09-07
3343,Web Developer,2026-09-01
470,Data Collection SW Engineer / Graduate,2026-09-01
667,"DevSecOps Engineer (Jenkins, Python)",2026-02-15


In [11]:
invalid_start_dates = df[
    df["start_date"].notna()
    & (df["start_date"] != "ASAP")
    & ~df["start_date"].str.match(
        r"^\d{4}-\d{2}-\d{2}$",
        na=False
    )
]

print("Unexpected start_date values:", len(invalid_start_dates))

invalid_start_dates[
    ["title", "start_date"]
].head(20)

Unexpected start_date values: 0


,title,start_date


## 5. Date columns

Convert date-related columns to datetime format and preserve the ASAP information separately.

In [12]:
df["valid_until"] = pd.to_datetime(
    df["valid_until"],
    format="%d.%m.%Y",
    errors="coerce"
)

df["scraped_at"] = pd.to_datetime(
    df["scraped_at"],
    errors="coerce"
)

df["start_asap"] = (
    df["start_date"]
    .eq("ASAP")
)

df["start_date_parsed"] = pd.to_datetime(
    df["start_date"].where(
        df["start_date"] != "ASAP"
    ),
    format="%Y-%m-%d",
    errors="coerce"
)

df[
    [
        "start_date",
        "start_asap",
        "start_date_parsed",
        "valid_until",
        "scraped_at",
    ]
].head(20)

,start_date,start_asap,start_date_parsed,valid_until,scraped_at
0,2026-09-07,False,2026-09-07,2026-09-20,2026-08-24 01:15:58
1,ASAP,True,NaT,2026-09-11,2026-08-24 01:16:02
2,ASAP,True,NaT,2026-09-06,2026-08-24 01:16:05
3,ASAP,True,NaT,2026-09-03,2026-08-24 01:16:09
4,ASAP,True,NaT,2026-09-16,2026-08-24 01:16:12
5,ASAP,True,NaT,2026-09-17,2026-08-24 01:16:16
6,ASAP,True,NaT,2026-08-31,2026-08-24 01:16:19
7,ASAP,True,NaT,2026-08-28,2026-08-24 01:16:23
8,NaN,False,NaT,2026-09-15,2026-08-24 01:16:26
9,ASAP,True,NaT,2026-08-29,2026-08-24 01:16:30


In [13]:
print("valid_until NaT:", df["valid_until"].isna().sum())
print("scraped_at NaT:", df["scraped_at"].isna().sum())

print()
print("ASAP:", df["start_asap"].sum())
print("Parsed start dates:", df["start_date_parsed"].notna().sum())
print("Missing original start_date:", df["start_date"].isna().sum())

print()
print(
    "Total start_date:",
    df["start_asap"].sum()
    + df["start_date_parsed"].notna().sum()
    + df["start_date"].isna().sum()
)

valid_until NaT: 63
scraped_at NaT: 0

ASAP: 2910
Parsed start dates: 282
Missing original start_date: 168

Total start_date: 3360


In [14]:
df["company_size"].value_counts(dropna=False).head(30)

company_size
NaN          886
500 - 999    420
1000+        405
250 - 499    287
100+         280
50 - 249     164
40+          105
+1000         95
500           75
1500          58
10 - 49       50
+6000         49
2000+         31
250           30
350+          26
13400+        26
7500+         25
500+          25
1700+         23
300+          23
200+          20
2400+         19
450+          16
5000          15
50+           14
110000        12
8000+         12
20+           11
5000+         10
60+           10
Name: count, dtype: int64

In [15]:
def company_size_format(value):

    if pd.isna(value):
        return "missing"

    value = str(value).strip()
    value = value.replace(",", "")
    value = value.replace(" ", "")

    if re.fullmatch(r"\d+\s*-\s*\d+", value):
        return "range"

    if re.fullmatch(r"\d+\+", value):
        return "min_plus"

    if re.fullmatch(r"\+\d+", value):
        return "plus_min"

    if re.fullmatch(r">\d+", value):
        return "greater_than"

    if re.fullmatch(r"\d+", value):
        return "exact"

    return "other"


df["company_size"].apply(
    company_size_format
).value_counts()

company_size
min_plus        1135
range            936
missing          886
exact            227
plus_min         165
greater_than      11
Name: count, dtype: int64

In [16]:
other_company_sizes = df[
    df["company_size"].apply(company_size_format) == "other"
]["company_size"].value_counts()

other_company_sizes

Series([], Name: count, dtype: int64)

## 6. Company size

Standardize company size values, extract employee-count bounds, and create consistent company-size segments. Open-ended values are assigned using their reported lower bound. The segments follow the source ranges: Small (below 50), Medium (50–249), Large (250–999), and Enterprise (1000+ employees).

In [17]:
def parse_company_size(value):

    if pd.isna(value):
        return np.nan, np.nan

    value = str(value).strip()

    value = value.replace(",", "")
    value = value.replace(" ", "")

    match = re.fullmatch(r"(\d+)-(\d+)", value)

    if match:
        return (
            int(match.group(1)),
            int(match.group(2)),
        )

    match = re.fullmatch(r"(\d+)\+", value)

    if match:
        return int(match.group(1)), np.nan

    match = re.fullmatch(r"\+(\d+)", value)

    if match:
        return int(match.group(1)), np.nan

    match = re.fullmatch(r">(\d+)", value)

    if match:
        return int(match.group(1)) + 1, np.nan

    if re.fullmatch(r"\d+", value):
        number = int(value)
        return number, number

    return np.nan, np.nan

In [18]:
df[
    ["company_size_min", "company_size_max"]
] = df["company_size"].apply(
    lambda x: pd.Series(
        parse_company_size(x)
    )
)

df[
    [
        "company_size",
        "company_size_min",
        "company_size_max",
    ]
].head(20)

,company_size,company_size_min,company_size_max
0,250 - 499,250.0,499.0
1,200+,200.0,NaN
2,5000,5000.0,5000.0
3,NaN,NaN,NaN
4,NaN,NaN,NaN
5,NaN,NaN,NaN
6,50 - 249,50.0,249.0
7,50-100,50.0,100.0
8,NaN,NaN,NaN
9,NaN,NaN,NaN


In [19]:
unparsed_company_sizes = df.loc[
    df["company_size"].notna()
    & df["company_size_min"].isna(),
    "company_size",
].value_counts()

unparsed_company_sizes

Series([], Name: count, dtype: int64)

In [20]:
print(
    "Original company size available:",
    df["company_size"].notna().sum()
)

print(
    "Parsed company size:",
    df["company_size_min"].notna().sum()
)

Original company size available: 2474
Parsed company size: 2474


In [21]:
company_size_order = [
    "Small",
    "Medium",
    "Large",
    "Enterprise",
    "Missing",
]

df["company_size_segment"] = pd.cut(
    df["company_size_min"],
    bins=[0, 50, 250, 1000, np.inf],
    labels=company_size_order[:-1],
    right=False,
)

df["company_size_segment"] = (
    df["company_size_segment"]
    .cat.add_categories(["Missing"])
    .fillna("Missing")
)

df["company_size_segment"].value_counts(
    sort=False
)

company_size_segment
Small         181
Medium        544
Large         929
Enterprise    820
Missing       886
Name: count, dtype: int64

## 7. Numerical data types

Convert integer-like numerical columns to nullable integer format.

In [22]:
df["experience_years_min"] = (
    df["experience_years_min"]
    .astype("Int64")
)

df["company_founded"] = (
    df["company_founded"]
    .astype("Int64")
)

df["company_size_min"] = (
    df["company_size_min"]
    .astype("Int64")
)

df["company_size_max"] = (
    df["company_size_max"]
    .astype("Int64")
)

df[
    [
        "experience_years_min",
        "company_founded",
        "company_size_min",
        "company_size_max",
    ]
].dtypes

experience_years_min    Int64
company_founded         Int64
company_size_min        Int64
company_size_max        Int64
dtype: object

## 8. Text and categorical standardization

Remove leading and trailing whitespace, convert empty strings to missing values, consolidate confirmed company-name aliases, and standardize job locations used in the analysis. Missing locations are classified as `Remote` only when the workplace model explicitly identifies remote work.

In [23]:
text_columns = df.select_dtypes(
    include="object"
).columns

for col in text_columns:
    df[col] = (
        df[col]
        .str.strip()
        .replace("", pd.NA)
    )

company_mapping = {
    "Allegro sp. z o.o.": "Allegro",
    "DCG sp. z o.o.": "DCG",
    "Finture Sp. z o.o": "Finture",
    "Finture Sp. z o.o.": "Finture",
    "Iteamly": "iTeamly",
    "UNITY-T GROUP": "Unity-T Group",
    "Comscore (via CC)": "Comscore via CC",
}

company_alias_mask = df["company"].isin(company_mapping)
df["company"] = df["company"].replace(company_mapping)
df["job_locations"] = df["job_locations"].replace(
    "Warsaw, Poland",
    "Warszawa",
)

remote_without_location = (
    df["job_locations"].isna()
    & df["workplace"].eq("Remote")
)

df.loc[remote_without_location, "job_locations"] = "Remote"
df["job_locations"] = df["job_locations"].fillna("Missing")

print("Text columns cleaned:", len(text_columns))
print("Company names standardized:", int(company_alias_mask.sum()))
print("Remote locations inferred:", int(remote_without_location.sum()))

Text columns cleaned: 19
Company names standardized: 44
Remote locations inferred: 1177


In [24]:
empty_strings = (
    df[text_columns]
    .eq("")
    .sum()
    .sum()
)

print("Empty strings remaining:", empty_strings)

Empty strings remaining: 0


## 9. Experience standardization

Standardize experience levels using job titles and retain only experience-year values confirmed by the requirements text.


In [25]:
def clean_experience(row):

    title = str(row["title"]).lower()

    levels_found = []

    if re.search(r"\b(junior|jr)\b", title):
        levels_found.append("Junior")

    if re.search(r"\b(mid|regular)\b", title):
        levels_found.append("Mid")

    if re.search(r"\b(senior|sr)\b", title):
        levels_found.append("Senior")

    if re.search(r"\b(expert|principal|head)\b", title):
        levels_found.append("Expert")

    if len(set(levels_found)) == 1:
        return levels_found[0]

    return row["experience"]


df["experience_clean"] = df.apply(
    clean_experience,
    axis=1
)

In [26]:
changed_mask = (
    df["experience"].fillna("Missing")
    !=
    df["experience_clean"].fillna("Missing")
)

print(
    "Changed experience levels:",
    changed_mask.sum()
)

Changed experience levels: 146


In [27]:
df["experience_clean"].value_counts(
    dropna=False
)

experience_clean
Senior    1816
Mid       1230
Expert     144
Junior     142
NaN         28
Name: count, dtype: int64

In [28]:
def extract_experience_years(text):

    if pd.isna(text):
        return pd.NA

    text = str(text).lower()

    patterns = [
  
        (
            r"(?:minimum|min\.?|co najmniej|od)?\s*"
            r"(\d{1,2})"
            r"\s*"
            r"(?:(?:[-–]|do)\s*\d{1,2}\+?)?"
            r"\s*\+?"
            r"\s*"
            r"(?:lat|lata|rok|roku)"
            r"[^.!?\n]{0,50}"
            r"doświadczen"
        ),


        (
            r"(?:minimum|min\.?|at least)?\s*"
            r"(\d{1,2})"
            r"\s*"
            r"(?:(?:[-–]|to)\s*\d{1,2}\+?)?"
            r"\s*\+?"
            r"\s*"
            r"(?:years?|yrs?)"
            r"[^.!?\n]{0,50}"
            r"experience"
        ),

        (
            r"experience"
            r"[^.!?\n]{0,30}"
            r"(\d{1,2})"
            r"\s*\+?"
            r"\s*(?:years?|yrs?)"
        ),
    ]

    for pattern in patterns:
        match = re.search(
            pattern,
            text,
            re.IGNORECASE
        )

        if match:
            return int(match.group(1))

    return pd.NA

In [29]:
df["experience_years_extracted"] = (
    df["requirements"]
    .apply(extract_experience_years)
    .astype("Int64")
)

In [30]:
df["experience_years_confident"] = (
    df["experience_years_min"]
    .where(
        df["experience_years_extracted"].notna()
        & (
            df["experience_years_min"]
            == df["experience_years_extracted"]
        )
    )
    .astype("Int64")
)

In [31]:
df = df.drop(
    columns=["experience_years_extracted"]
)

In [32]:
df = df.drop(columns=["experience"])

df = df.rename(
    columns={
        "experience_clean": "experience"
    }
)

In [33]:
df["experience"].value_counts(dropna=False)

experience
Senior    1816
Mid       1230
Expert     144
Junior     142
NaN         28
Name: count, dtype: int64

In [34]:
df = df.drop(
    columns=[
        "experience_years_min",
    ],
    errors="ignore"
)

In [35]:
df = df.rename(
    columns={
        "experience_years_confident": "experience_years_min"
    }
)

In [36]:
print("Dataset shape:", df.shape)

print()
print(df["experience"].value_counts(dropna=False))

print()
print(
    "Experience years available:",
    df["experience_years_min"].notna().sum()
)

print(
    "Duplicate URLs:",
    df["url"].duplicated().sum()
)

print(
    "Missing titles:",
    df["title"].isna().sum()
)

Dataset shape: (3360, 30)

experience
Senior    1816
Mid       1230
Expert     144
Junior     142
NaN         28
Name: count, dtype: int64

Experience years available: 1793
Duplicate URLs: 0
Missing titles: 0


## 10. Salary validation and normalization

Remove salary values whose period cannot be identified, then convert complete salary ranges to approximate monthly PLN values. Source salary columns are retained, while normalized values are stored in separate columns.

Assumptions:

- 168 working hours per month,
- 21 working days per month,
- 12 months per year,
- PLN exchange rate equal to 1,
- EUR and USD converted using NBP average exchange rates from table A dated 2026-08-21.

Fixed rates and a documented date are used to keep the analysis reproducible. Successfully normalized offers are marked with `salary_normalization_eligible`. For salary comparisons, normalized midpoints below PLN 5,000 or above PLN 100,000 are treated as implausible source-data values and marked as ineligible without removing the original records.


In [37]:
salary_columns = [
    "salary_min",
    "salary_max",
    "salary_currency",
]

missing_salary_period = df["salary_period"].isna()
salary_rows_cleared = (
    missing_salary_period
    & df[salary_columns].notna().any(axis=1)
).sum()

df.loc[
    missing_salary_period,
    salary_columns,
] = pd.NA

print("Salary rows cleared because period is missing:", salary_rows_cleared)


Salary rows cleared because period is missing: 1


In [38]:
df["salary_min"] = df["salary_min"].astype("Int64")
df["salary_max"] = df["salary_max"].astype("Int64")

df[
    [
        "salary_min",
        "salary_max",
        "experience_years_min",
        "company_founded",
        "company_size_min",
        "company_size_max",
    ]
].dtypes

salary_min              Int64
salary_max              Int64
experience_years_min    Int64
company_founded         Int64
company_size_min        Int64
company_size_max        Int64
dtype: object

In [39]:
# Salary normalization assumptions
HOURS_PER_MONTH = 168
DAYS_PER_MONTH = 21
MONTHS_PER_YEAR = 12
SALARY_MONTHLY_MIN = 5_000
SALARY_MONTHLY_MAX = 100_000

# NBP average exchange rates, table A, 2026-08-21
EXCHANGE_RATE_DATE = "2026-08-21"

currency_to_pln = {
    "PLN": 1.0,
    "EUR": 4.3122,
    "USD": 3.6839,
}

period_to_monthly = {
    "hour": HOURS_PER_MONTH,
    "day": DAYS_PER_MONTH,
    "month": 1.0,
    "year": 1 / MONTHS_PER_YEAR,
}

In [40]:
df["salary_exchange_rate_to_pln"] = (
    df["salary_currency"]
    .map(currency_to_pln)
)

currency_factor = df["salary_exchange_rate_to_pln"]

period_factor = (
    df["salary_period"]
    .map(period_to_monthly)
)

normalization_factor = (
    currency_factor
    * period_factor
)

In [41]:
df["salary_mid"] = (
    df["salary_min"]
    + df["salary_max"]
) / 2

In [42]:
df["salary_min_pln_monthly"] = (
    df["salary_min"]
    * normalization_factor
)

df["salary_max_pln_monthly"] = (
    df["salary_max"]
    * normalization_factor
)

df["salary_mid_pln_monthly"] = (
    df["salary_mid"]
    * normalization_factor
)

In [43]:
normalized_salary_columns = [
    "salary_min_pln_monthly",
    "salary_max_pln_monthly",
    "salary_mid_pln_monthly",
]

df[normalized_salary_columns] = (
    df[normalized_salary_columns]
    .round(2)
)

In [44]:
df["salary_normalization_eligible"] = (
    df["salary_min_pln_monthly"].notna()
    & df["salary_max_pln_monthly"].notna()
    & df["salary_mid_pln_monthly"].notna()
)

df["salary_analysis_eligible"] = (
    df["salary_normalization_eligible"]
    & df["salary_mid_pln_monthly"].between(
        SALARY_MONTHLY_MIN,
        SALARY_MONTHLY_MAX,
    )
)

In [45]:
foreign_currency = df["salary_currency"].isin(
    ["EUR", "USD"]
)

df["salary_exchange_rate_date"] = pd.NaT

df.loc[
    foreign_currency,
    "salary_exchange_rate_date",
] = pd.Timestamp(EXCHANGE_RATE_DATE)

## 11. Final quality checks

Validate the prepared dataset before saving it for exploratory analysis.


In [46]:
salary_summary_columns = [
    "salary_min_pln_monthly",
    "salary_max_pln_monthly",
    "salary_mid_pln_monthly",
]

assert df["job_id"].is_unique
assert df["url"].is_unique
assert not (
    df["salary_min_pln_monthly"]
    > df["salary_max_pln_monthly"]
).fillna(False).any()
assert df.loc[
    df["salary_period"].isna(),
    ["salary_min", "salary_max", "salary_currency"],
].isna().all().all()
assert not (
    df["salary_analysis_eligible"]
    & ~df["salary_normalization_eligible"]
).any()

print("Final number of rows:", len(df))
print("Unique offers:", df["url"].nunique())
print(
    "Offers eligible for salary normalization:",
    int(df["salary_normalization_eligible"].sum()),
)
print(
    "Offers eligible for salary analysis:",
    int(df["salary_analysis_eligible"].sum()),
)

display(
    df.loc[
        df["salary_analysis_eligible"],
        salary_summary_columns,
    ].describe().round(2)
)


Final number of rows: 3360
Unique offers: 3360
Offers eligible for salary normalization: 2352
Offers eligible for salary analysis: 2339


,salary_min_pln_monthly,salary_max_pln_monthly,salary_mid_pln_monthly
count,2339.0,2339.0,2339.0
mean,22089.7,27317.49,24703.59
std,6824.43,8160.47,7331.91
min,4500.0,5500.0,5000.0
25%,18000.0,22000.0,20160.0
50%,21840.0,26880.0,24360.0
75%,26250.0,31920.0,29030.0
max,63000.0,100800.0,81900.0


In [47]:
pd.crosstab(
    df.loc[df["salary_analysis_eligible"], "salary_period"],
    df.loc[df["salary_analysis_eligible"], "salary_currency"],
    margins=True,
)


salary_currency,EUR,PLN,USD,All
salary_period,,,,
day,1,335,0,336
hour,20,1122,6,1148
month,7,795,9,811
year,7,37,0,44
All,35,2289,15,2339


## 12. Save the cleaned dataset

Save the fully prepared dataset only after all transformations and validation checks have completed.


In [48]:
PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CLEAN_PATH = (
    PROCESSED_DIR
    / "nofluff_it_jobs_clean.csv"
)

In [49]:
df.to_csv(
    CLEAN_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Clean dataset saved to:", CLEAN_PATH)
print("Saved rows:", len(df))


Clean dataset saved to: G:\pandas\job_market_intelligence\data\processed\nofluff_it_jobs_clean.csv
Saved rows: 3360
